<a href="https://colab.research.google.com/github/mainapkago-pixel/course-project-GitHub-/blob/main/Week_5_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd

df = pd.read_csv("/content/stations data.csv")

# Print available columns to diagnose the exact name of the target column
print("Available columns in dataset:", df.columns.tolist())

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Set the target column to match the column name in the CSV
target = "Compliant"

if target in df.columns:
    # Remove rows with missing target values
    df = df.dropna(subset=[target]).copy()

    # Separate features (X) and target (y)
    X = df.drop(columns=[target])
    y = df[target]

    # Remove identifiers if present
    X = X.drop(
        columns=["Station ID", "Station Name"],
        errors="ignore"
    )

    # Identify numerical and categorical features
    numeric_features = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    # Prepare numerical features
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # Prepare categorical features
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine preprocessing steps
    preprocessor = ColumnTransformer([
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ])

    # Split into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training records:", len(X_train))
    print("Testing records:", len(X_test))
    print("Features used:", X.columns.tolist())
    print("Target classes:", y.unique())
else:
    print(f"\nERROR: Could not find '{target}' in the dataset columns. Please update the 'target' variable with one of the printed names above.")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Create Logistic Regression model
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Create Random Forest model
forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train both models
logistic_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Both models trained successfully.")
# Predict compliance on the test dataset
logistic_predictions = logistic_model.predict(X_test)
forest_predictions = forest_model.predict(X_test)

# Display actual and predicted values
results = pd.DataFrame({
    "Actual Compliance": y_test.to_numpy(),
    "Logistic Regression Prediction": logistic_predictions,
    "Random Forest Prediction": forest_predictions
})

print(results.head(10).to_string(index=False))


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Function to evaluate a model
def evaluate_model(name, actual, predicted):
    print("\n", name)
    print("Accuracy:", round(
        accuracy_score(actual, predicted), 3
    ))
    print("Precision:", round(
        precision_score(
            actual, predicted,
            average="weighted",
            zero_division=0
        ), 3
    ))
    print("Recall:", round(
        recall_score(
            actual, predicted,
            average="weighted",
            zero_division=0
        ), 3
    ))
    print("F1-score:", round(
        f1_score(
            actual, predicted,
            average="weighted",
            zero_division=0
        ), 3
    ))

    print("\nClassification Report:")
    print(classification_report(
        actual, predicted,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(actual, predicted))


# Evaluate Logistic Regression
evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_predictions
)

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, forest_predictions)
    ],
    "Precision": [
        precision_score(
            y_test, logistic_predictions,
            average="weighted", zero_division=0
        ),
        precision_score(
            y_test, forest_predictions,
            average="weighted", zero_division=0
        )
    ],
    "Recall": [
        recall_score(
            y_test, logistic_predictions,
            average="weighted", zero_division=0
        ),
        recall_score(
            y_test, forest_predictions,
            average="weighted", zero_division=0
        )
    ],
    "F1-score": [
        f1_score(
            y_test, logistic_predictions,
            average="weighted", zero_division=0
        ),
        f1_score(
            y_test, forest_predictions,
            average="weighted", zero_division=0
        )
    ]
})

Available columns in dataset: ['Station ID', 'County', 'Region', 'PMS Price (KES/L)', 'AGO Price (KES/L)', 'Tank Condition', 'Fire Extinguishers', 'Drainage', 'Earthing', 'Spill Kit', 'Compliant']
Training records: 24
Testing records: 6
Features used: ['County', 'Region', 'PMS Price (KES/L)', 'AGO Price (KES/L)', 'Tank Condition', 'Fire Extinguishers', 'Drainage', 'Earthing', 'Spill Kit']
Target classes: ['Yes' 'No']
Both models trained successfully.
Actual Compliance Logistic Regression Prediction Random Forest Prediction
               No                             No                       No
              Yes                            Yes                      Yes
               No                             No                       No
              Yes                            Yes                      Yes
               No                             No                       No
              Yes                            Yes                      Yes

 Logistic Regression
Accur

Classification or Regression Problem
This project is a classification problem because the aim is to predict whether a petrol station complies with Kenyan standards or does not comply. The target variable, Compliance, has two possible categories: 1 (compliant) and 0 (non-compliant). The model will learn from previous inspection data to classify petrol stations based on their inspection findings.

### Training and Testing Split

The dataset is divided into 80% training data and 20% testing data. The training set is used to train the machine learning models, while the testing set is used to evaluate their ability to predict compliance for stations they have not seen before. A random state of 42 makes the split reproducible.

### Model Selection

Two supervised classification models are selected: Logistic Regression and Random Forest Classifier. Logistic Regression provides a simple baseline for predicting compliance, while Random Forest combines multiple decision trees to identify patterns in the inspection data.

### Model Evaluation

The models are evaluated using accuracy, precision, recall, and F1-score. Accuracy measures the proportion of correct predictions. Precision measures how reliable positive predictions are, recall measures how many actual positive cases are identified, and the F1-score balances precision and recall. These metrics help determine how effectively each model predicts petrol station compliance.